# NB5 - Generate V_lit Vectors (Person C)
**Run this AFTER NB3 is complete and `plutchik-model-v2` is shared as a Kaggle dataset.**

## Setup:
1. `+ Add Data` -> search `plutchik-model-v2` -> add it
2. Add sarcasm data: `+ Add Data` -> search `mustard` (or upload a MUSTARD CSV)
3. Enable GPU: Settings -> Accelerator -> T4 x1
4. Run all cells

Output: `mentalmanip_vlits.csv` -> share as Kaggle dataset `mentalmanip-vlits`
This export now contains MentalManip rows plus MUSTARD sarcasm rows.
Person B adds this as input to NB6.


## 0. Install

In [ ]:
%%capture
!pip install transformers accelerate datasets -q
print('done')

## 1. Config

In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Model path — handles both Kaggle dataset upload styles
_candidates = [
    '/kaggle/input/datasets/bobhendriks/plutchik-model-v2',   # owner-prefixed upload
    '/kaggle/input/plutchik-model-v2',                        # direct dataset name
]
MODEL_DIR = None
for _base in _candidates:
    if os.path.isdir(_base):
        for root, dirs, files in os.walk(_base):
            if 'model_config.json' in files:
                MODEL_DIR = root
                break
    if MODEL_DIR:
        break
if MODEL_DIR is None:
    raise FileNotFoundError(
        f'Could not find model_config.json under any of: {_candidates}'
    )
print(f'MODEL_DIR: {MODEL_DIR}')

OUT_DIR    = '/kaggle/working'
EMOTIONS   = ['joy','trust','fear','surprise','sadness','disgust','anger','anticipation']
MAX_LEN    = 128
BATCH_SIZE = 32
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Rebuild Model 1 Architecture (PlutchikModelV2 — must match NB3 exactly)

In [ ]:
from transformers import AutoTokenizer, AutoModel


class EmotionAttentionBlock(nn.Module):
    def __init__(self, hidden_size, n_heads=4, out_dim=128):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = hidden_size // n_heads
        self.scale    = self.head_dim ** -0.5
        self.query    = nn.Parameter(torch.randn(1, 1, hidden_size) * 0.02)
        self.k_proj   = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj   = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, out_dim)
        self.norm     = nn.LayerNorm(out_dim)
        self.dropout  = nn.Dropout(0.1)

    def forward(self, hidden_states, attention_mask):
        B, L, H = hidden_states.shape
        nh, hd  = self.n_heads, self.head_dim
        q = self.query.expand(B, -1, -1)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)
        q = q.view(B, 1, nh, hd).transpose(1, 2)
        k = k.view(B, L, nh, hd).transpose(1, 2)
        v = v.view(B, L, nh, hd).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        if attention_mask is not None:
            pad = (attention_mask == 0).unsqueeze(1).unsqueeze(2)
            attn = attn.masked_fill(pad, float('-inf'))
        attn = self.dropout(torch.softmax(attn, dim=-1))
        out  = (attn @ v).squeeze(2).reshape(B, H)
        return self.norm(self.out_proj(out))


class PlutchikModelV2(nn.Module):
    def __init__(self, base_model_name, n_emotions=8, out_dim=128, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        H = self.encoder.config.hidden_size
        self.emotion_blocks = nn.ModuleList([EmotionAttentionBlock(H, 4, out_dim) for _ in range(n_emotions)])
        self.emotion_heads  = nn.ModuleList([
            nn.Sequential(nn.Linear(out_dim, 32), nn.GELU(), nn.Dropout(dropout), nn.Linear(32, 1))
            for _ in range(n_emotions)
        ])
        self.temperature = nn.Parameter(torch.full((n_emotions,), 1.0))
        self.conf_head   = nn.Sequential(nn.Linear(H, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 1), nn.Sigmoid())
        self.cls_head    = nn.Sequential(nn.Linear(H, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, n_emotions))
        self.drop = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        enc  = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        h    = self.drop(enc.last_hidden_state)
        cls  = h[:, 0]
        logits = [head(block(h, attention_mask)) for block, head in zip(self.emotion_blocks, self.emotion_heads)]
        logits = torch.cat(logits, dim=-1)
        temp   = torch.clamp(self.temperature, 0.5, 5.0)
        scores = torch.sigmoid(logits * temp)
        return scores, self.conf_head(cls).squeeze(-1), self.cls_head(cls)


with open(os.path.join(MODEL_DIR, 'model_config.json')) as f:
    cfg = json.load(f)

print('Loading tokenizer + model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = PlutchikModelV2(cfg['base_model']).to(DEVICE).float()
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_model.pt'), map_location=DEVICE))
model.eval()
print(f'Model loaded | Best val Spearman: {cfg["best_val_spearman"]:.4f}')

## 3. Batch Inference Helper

In [ ]:
def get_vlit_batch(texts):
    """Returns np.ndarray shape (N, 8) — one Plutchik vector per text."""
    all_scores = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        enc = tokenizer(
            batch, max_length=MAX_LEN, padding='max_length',
            truncation=True, return_tensors='pt'
        ).to(DEVICE)
        with torch.no_grad():
            scores, _, _ = model(
                enc['input_ids'], enc['attention_mask'],
                enc.get('token_type_ids'),
            )
        all_scores.append(scores.float().cpu().numpy())
        if (i // BATCH_SIZE) % 10 == 0:
            print(f'  Processed {min(i + BATCH_SIZE, len(texts)):,} / {len(texts):,}')
    return np.concatenate(all_scores, axis=0)


print('get_vlit_batch() ready')

## 4. Load MentalManip + MUSTARD Datasets


In [ ]:
from datasets import load_dataset

print('Loading MentalManip...')

try:
    ds = load_dataset('audreyeleven/MentalManip', 'mentalmanip_detailed', split='train')
    print(f'Loaded MentalManip from HuggingFace (mentalmanip_detailed): {len(ds):,} rows')
except Exception as e:
    print(f'MentalManip HuggingFace load failed ({e}), trying local CSV...')
    csv_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.lower().endswith('.csv') and 'mentalmanip' in f.lower():
                csv_path = os.path.join(root, f)
                break
        if csv_path:
            break
    if csv_path is None:
        raise FileNotFoundError('Could not find a MentalManip CSV anywhere under /kaggle/input')
    print(f'Using local MentalManip CSV: {csv_path}')
    ds = load_dataset('csv', data_files={'train': csv_path})['train']
    print(f'Loaded MentalManip from CSV: {len(ds):,} rows')

df = ds.to_pandas()
print(f'MentalManip columns: {list(df.columns)}')
print(df.head(3))

# ── Sarcasm datasets (priority order) ───────────────────────────────────────
# We load from multiple sources so we get enough diverse, high-quality sarcasm
# examples. All sources are unified into sarcasm_df with columns [text, source_dataset].
print('\nLoading sarcasm datasets...')
_sarcasm_dfs = []

# 1. tweet_eval irony — conversational tweets, best domain match
try:
    _te = load_dataset('tweet_eval', 'irony', trust_remote_code=True)
    _te_df = pd.concat([_te[s].to_pandas() for s in _te.keys()], ignore_index=True)
    _te_sar = _te_df[_te_df['label'] == 1][['text']].copy()
    _te_sar['source_dataset'] = 'tweet_eval_irony'
    _sarcasm_dfs.append(_te_sar)
    print(f'  tweet_eval irony  : {len(_te_sar):,} examples')
except Exception as e:
    print(f'  tweet_eval irony failed: {e}')

# 2. Sarcasm news headlines — high label quality, different writing style
for _ds_name in ['raquiba/Sarcasm_News_Headline', 'MidhunKanadan/sarcasm-detection']:
    try:
        _sh = load_dataset(_ds_name, trust_remote_code=True)
        _sh_df = pd.concat([_sh[s].to_pandas() for s in _sh.keys()], ignore_index=True)
        _lbl = next((c for c in _sh_df.columns if any(k in c.lower() for k in ['sarcas','label','irony'])), None)
        _txt = next((c for c in _sh_df.columns if any(k in c.lower() for k in ['headline','text','sentence','utterance'])), None)
        if _lbl and _txt:
            _sh_sar = _sh_df[pd.to_numeric(_sh_df[_lbl], errors='coerce') == 1][[_txt]].copy()
            _sh_sar.columns = ['text']
            _sh_sar['source_dataset'] = 'sarcasm_headlines'
            _sarcasm_dfs.append(_sh_sar)
            print(f'  {_ds_name}: {len(_sh_sar):,} examples')
            break
    except Exception as e:
        print(f'  {_ds_name} failed: {e}')

# 3. MUSTARD — TV-show dialogue sarcasm (context-dependent, good supplement)
def _find_local_csv(keyword):
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.lower().endswith('.csv') and keyword in f.lower():
                return os.path.join(root, f)
    return None

for _mode, _name in [('csv', None), ('hf', 'Yaxin/MUSTARD'), ('hf', 'jhamel/mustard'), ('hf', 'tasksource/mustard')]:
    try:
        if _mode == 'csv':
            _csv = _find_local_csv('mustard')
            if _csv is None:
                raise FileNotFoundError('no local mustard CSV')
            _m_df = load_dataset('csv', data_files={'train': _csv})['train'].to_pandas()
        else:
            _m_ds = load_dataset(_name, trust_remote_code=True)
            _m_df = pd.concat([_m_ds[s].to_pandas() for s in _m_ds.keys()], ignore_index=True)
        _txt = next((c for c in _m_df.columns if any(k in c.lower() for k in ['utterance','text','sentence'])), None)
        _lbl = next((c for c in _m_df.columns if any(k in c.lower() for k in ['label','sarcasm','sarcastic'])), None)
        if _txt:
            if _lbl:
                _m_df = _m_df[pd.to_numeric(_m_df[_lbl], errors='coerce') == 1]
            _m_sar = _m_df[[_txt]].copy()
            _m_sar.columns = ['text']
            _m_sar['source_dataset'] = 'mustard'
            _sarcasm_dfs.append(_m_sar)
            print(f'  MUSTARD [{_mode}:{_name}]: {len(_m_sar):,} examples')
            break
    except Exception as e:
        print(f'  MUSTARD [{_mode}:{_name}] failed: {e}')

if not _sarcasm_dfs:
    raise RuntimeError(
        'Could not load any sarcasm dataset. '
        'Add at least one of: tweet_eval irony, raquiba/Sarcasm_News_Headline, or a MUSTARD CSV.'
    )

sarcasm_df = pd.concat(_sarcasm_dfs, ignore_index=True)
sarcasm_df['text'] = sarcasm_df['text'].astype(str).str.strip()
sarcasm_df = sarcasm_df[sarcasm_df['text'].ne('')].drop_duplicates(subset='text').reset_index(drop=True)
print(f'\nTotal sarcasm examples (deduplicated): {len(sarcasm_df):,}')
print(sarcasm_df['source_dataset'].value_counts())


## 5. Normalise Column Names
MentalManip detailed schema:
- `dialogue` -> utterance text
- `dialogue_id`, `turn_index`, `speaker` are reconstructed for conversation-aware downstream features
- `source_dataset` tracks whether a row came from MentalManip or MUSTARD


In [ ]:
cols = list(df.columns)
print('MentalManip columns:', cols)

# Annotator columns are suffixed _1/_2/_3
im_idx   = [i for i, c in enumerate(cols) if c.startswith('manipulative_')]
tech_idx = [i for i, c in enumerate(cols) if c.startswith('technique_')]
vuln_idx = [i for i, c in enumerate(cols) if c.startswith('vulnerability_')]

im_df   = df.iloc[:, im_idx].apply(pd.to_numeric, errors='coerce').fillna(0)
tech_df = df.iloc[:, tech_idx]
vuln_df = df.iloc[:, vuln_idx]

# annotator_votes: how many of the 3 annotators flagged this as manipulative
# Rows with 1 vote = genuine annotator disagreement → Ambiguous in NB6
# Rows with 0 votes = clearly genuine
# Rows with ≥2 votes = clear manipulation
annotator_votes = im_df.sum(axis=1).astype(int)
is_manip        = (annotator_votes >= 2).astype(int)

technique = tech_df.apply(
    lambda r: next((v for v in r if pd.notna(v) and str(v).strip() not in ('', 'nan')), ''),
    axis=1
)
vulnerability = vuln_df.apply(
    lambda r: next((v for v in r if pd.notna(v) and str(v).strip() not in ('', 'nan')), ''),
    axis=1
)

mental_df = pd.DataFrame({
    'dialogue_id':     df.iloc[:, cols.index('inner_id')].values,
    'turn_index':      0,
    'speaker':         'unknown',
    'text':            df.iloc[:, cols.index('dialogue')].values,
    'is_manipulative': is_manip.values,
    'technique':       technique.values,
    'vulnerability':   vulnerability.values,
    'annotator_votes': annotator_votes.values,
    'source_dataset':  'mentalmanip',
})

mental_df = mental_df.dropna(subset=['text']).reset_index(drop=True)
print(f'MentalManip clean rows: {len(mental_df):,}')
print(f'Annotator vote distribution:')
for v, n in mental_df['annotator_votes'].value_counts().sort_index().items():
    label = {0: 'clear genuine', 1: 'ambiguous (disagreement)', 2: 'likely manipulation', 3: 'clear manipulation'}.get(v, '')
    print(f'  {v} votes ({label}): {n:,}')

# ── Build sarcasm_out from unified sarcasm_df ────────────────────────────────
max_id = int(mental_df['dialogue_id'].max()) + 1
sarcasm_out = sarcasm_df.copy()
sarcasm_out['dialogue_id']     = np.arange(len(sarcasm_out), dtype=np.int64) + max_id
sarcasm_out['turn_index']      = 0
sarcasm_out['speaker']         = 'sarcasm_speaker'
sarcasm_out['is_manipulative'] = 1
sarcasm_out['technique']       = 'sarcasm'
sarcasm_out['vulnerability']   = ''
sarcasm_out['annotator_votes'] = 3  # confirmed sarcasm → high confidence
sarcasm_out = sarcasm_out[
    ['dialogue_id','turn_index','speaker','text','is_manipulative',
     'technique','vulnerability','source_dataset','annotator_votes']
].reset_index(drop=True)
print(f'\nSarcasm rows: {len(sarcasm_out):,}')
print(sarcasm_out['source_dataset'].value_counts())

df = mental_df.copy()


## 5b. Parse Multi-Turn Dialogues into Individual Turn Rows
Each row in MentalManip contains a full dialogue snippet (`Person1: ...\nPerson2: ...`).
We split those snippets into turn-level rows so NB6 can compute conversation deltas.
MUSTARD rows are appended later as sarcasm examples with their own `source_dataset` tag.


In [ ]:
import re

# Guard: detect if text already lacks PersonN: prefixes (parse already ran)
_sample = df['text'].dropna().head(50)
_has_speaker = _sample.str.contains(r'Person\d+\s*:', regex=True).any()
if not _has_speaker:
    raise RuntimeError(
        "No 'PersonN:' patterns found in text -> parse cell may have already run. "
        "Re-run from the load cell (cell-10) before running this cell again."
    )

def parse_dialogue_turns(row):
    """Split a 'Person1: ...\nPerson2: ...' dialogue into individual turn rows."""
    text = str(row['text'])
    matches = list(re.finditer(r'(?:^|\n)(Person\d+)\s*:\s*', text))

    base = {
        'dialogue_id':     row['dialogue_id'],
        'is_manipulative': row['is_manipulative'],
        'technique':       row['technique'],
        'vulnerability':   row['vulnerability'],
        'source_dataset':  row.get('source_dataset', 'mentalmanip'),
        'annotator_votes': row.get('annotator_votes', 3),
    }

    if len(matches) <= 1:
        return [{
            **base,
            'turn_index': 0,
            'speaker':    matches[0].group(1) if matches else 'unknown',
            'text':       re.sub(r'^Person\d+\s*:\s*', '', text).strip(),
        }]

    turns = []
    for i, m in enumerate(matches):
        start     = m.end()
        end       = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        turn_text = text[start:end].strip()
        if turn_text:
            turns.append({**base, 'turn_index': i, 'speaker': m.group(1), 'text': turn_text})

    return turns if turns else [{**base, 'turn_index': 0, 'speaker': 'unknown', 'text': text}]


print('Parsing multi-turn MentalManip dialogues...')
n_before = len(df)
all_turns = []
for _, row in df.iterrows():
    all_turns.extend(parse_dialogue_turns(row))

df = pd.DataFrame(all_turns).reset_index(drop=True)
print(f'Rows before: {n_before:,}  ->  after: {len(df):,}')
print(f'Unique dialogues  : {df["dialogue_id"].nunique():,}')
print(f'Avg turns/dialogue: {len(df)/df["dialogue_id"].nunique():.1f}')
print(df[['dialogue_id','turn_index','speaker','text','is_manipulative','annotator_votes','source_dataset']].head(10))

COMBINED_COLS = ['dialogue_id','turn_index','speaker','text','is_manipulative',
                 'technique','vulnerability','source_dataset','annotator_votes']

if len(sarcasm_out):
    print(f'\nAppending sarcasm rows: {len(sarcasm_out):,}')
    df = pd.concat([df[COMBINED_COLS], sarcasm_out[COMBINED_COLS]], ignore_index=True)

df = df.sort_values(['dialogue_id', 'turn_index']).reset_index(drop=True)
print(f'\nCombined rows: {len(df):,}')
print(df['source_dataset'].value_counts(dropna=False))
print(f'annotator_votes dist: {dict(df["annotator_votes"].value_counts().sort_index())}')


In [ ]:
## --- Post-parse dialogue structure diagnostics ---
n_rows    = len(df)
n_dialogs = df['dialogue_id'].nunique()
avg_turns = n_rows / n_dialogs
print(f'Unique dialogue_ids : {n_dialogs:,}')
print(f'Total rows          : {n_rows:,}')
print(f'Avg turns/dialogue  : {avg_turns:.1f}')
print(df[df['source_dataset'] == 'mentalmanip'].groupby('dialogue_id').size().describe().round(1))

if n_dialogs == n_rows:
    print('\n[WARNING] Every row has a unique dialogue_id — context features in NB6 will all be zero!')
else:
    print(f'\n[OK] Multi-turn structure intact — NB6 context features will work correctly.')

print(f'\nannotator_votes dist (mentalmanip): ', end='')
mm = df[df['source_dataset'] == 'mentalmanip']
print(dict(mm['annotator_votes'].value_counts().sort_index()))
print(f'  -> Ambiguous candidates (1-vote): {(mm["annotator_votes"] == 1).sum():,}')


## 6. Run Model 1 Inference

In [ ]:
print(f'Running inference on {len(df):,} utterances...')
print(df['source_dataset'].value_counts(dropna=False))
texts = df['text'].astype(str).tolist()
vlits = get_vlit_batch(texts)   # shape (N, 8)
print(f'Done. V_lit matrix shape: {vlits.shape}')


vlit_df = pd.DataFrame(vlits, columns=EMOTIONS)

# Only include metadata columns that actually exist (robust to missing mappings)
META_COLS = ['dialogue_id', 'turn_index', 'speaker', 'text',
             'is_manipulative', 'technique', 'vulnerability']
meta_cols_present = [c for c in META_COLS if c in df.columns]
missing_meta = [c for c in META_COLS if c not in df.columns]
if missing_meta:
    print(f'[WARNING] Missing metadata columns (will be omitted): {missing_meta}')

out_df = pd.concat([
    df[meta_cols_present].reset_index(drop=True),
    vlit_df
], axis=1)

out_path = os.path.join(OUT_DIR, 'mentalmanip_vlits.csv')
out_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)')
print(f'Rows: {len(out_df):,}  |  Columns: {list(out_df.columns)}')
print(out_df.head(3))

In [ ]:
vlit_df = pd.DataFrame(vlits, columns=EMOTIONS)
out_df  = pd.concat([
    df[['dialogue_id','turn_index','speaker','text','is_manipulative',
        'technique','vulnerability','source_dataset','annotator_votes']].reset_index(drop=True),
    vlit_df
], axis=1)

out_path = os.path.join(OUT_DIR, 'mentalmanip_vlits.csv')
out_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)')
print(f'Rows: {len(out_df):,}  |  Columns: {list(out_df.columns)}')
print(out_df.head(3))


## 8. Sanity Check — Sample Predictions

In [ ]:
sample = out_df.sample(5, random_state=42)
for _, row in sample.iterrows():
    top2 = sorted([(e, row[e]) for e in EMOTIONS], key=lambda x: -x[1])[:2]
    dom  = '  '.join(f'{e}={v:.3f}' for e, v in top2)
    label = 'SARCASM' if row['source_dataset'] == 'mustard' else ('MANIP' if row['is_manipulative'] else 'ok')
    print(f'[{label}] {row["text"][:60]:<60} -> {dom}')


## 9. Zip & Download

In [ ]:
import shutil
zip_path = shutil.make_archive(os.path.join(OUT_DIR, 'mentalmanip_vlits'), 'zip',
                                OUT_DIR, 'mentalmanip_vlits.csv')
print(f'Zipped: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('Share mentalmanip_vlits.csv as Kaggle dataset: mentalmanip-vlits')
print('This export includes both MentalManip and MUSTARD rows.')
print('Person B adds it as input to NB6.')
